In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
import pyspark.sql.functions as F
import json

# =====================================================================
# TRẠM 1: TIỀN XỬ LÝ DỮ LIỆU & KHUẾCH ĐẠI HẠT GIỐNG (BOOTSTRAPPING)
# =====================================================================

print("🚀 ĐANG KHỞI ĐỘNG ĐỘNG CƠ BIG DATA (APACHE SPARK)...")
spark = SparkSession.builder \
    .appName("Music_Taste_Engine_Module2") \
    .config("spark.driver.memory", "12g") \
    .config("spark.sql.shuffle.partitions", "400") \
    .config("spark.default.parallelism", "400") \
    .config("spark.memory.fraction", "0.8") \
    .config("spark.local.dir", "/kaggle/working/spark_tmp") \
    .getOrCreate()

print("⏳ Đang đọc file và xử lý Dữ liệu bẩn + Collab...")
folder1 = '/kaggle/input/datasets/js042710/second3t1k/CLEARDATA/CLEARDATA'
folder2 = '/kaggle/input/datasets/js042710/second3t1k/data DM/data DM'

raw_df = spark.read.csv([folder1, folder2], header=True, inferSchema=True) \
              .select("user_id", "artist_name") \
              .filter(F.col("user_id").rlike("^\d+$")) \
              .filter(F.col("artist_name").isNotNull())

df_lower = raw_df.select(
    F.col("user_id").cast("integer"),
    F.lower(F.col("artist_name")).alias("artist_name")
).dropna(subset=["user_id", "artist_name"])

regex_pattern = r'\s*(?:&|feat\.?|ft\.?|/|;|\bx\b)\s*'
df_replaced = df_lower.withColumn("artist_name", F.regexp_replace("artist_name", regex_pattern, ","))
df_array = df_replaced.withColumn("artist_array", F.split("artist_name", ","))
df_exploded = df_array.withColumn("single_artist", F.explode("artist_array"))

clean_df = df_exploded.select(
    "user_id",
    F.trim(F.col("single_artist")).alias("artist_name")
).filter(F.col("artist_name") != "") \
 .repartition(400, "user_id") \
 .persist()
print("✅ Đã xử lý Collab xong!")

print("⏳ Đang nén dữ liệu thành Tập hợp (Set) theo User...")
user_artists_df = clean_df.groupBy("user_id").agg(
    F.collect_set("artist_name").alias("artist_set")
)
filtered_users_df = user_artists_df.filter(F.size(F.col("artist_set")) >= 5) \
                                   .persist()
print("✅ Đã nén thành Giỏ xong!")

# =====================================================================
# KHUẾCH ĐẠI HẠT GIỐNG
# =====================================================================
print("🚀 Đang kích hoạt Bootstrapping...")

seed_genres_f0 = {
    "POP": ["taylor swift", "ariana grande", "justin bieber", "ed sheeran"],
    "HIPHOP": ["eminem", "kendrick lamar", "drake", "travis scott"],
    "EDM": ["martin garrix", "avicii", "calvin harris", "skrillex"],
    "RNB": ["the weeknd", "sza", "frank ocean", "bruno mars"],
    "ROCK": ["linkin park", "queen", "ac/dc", "coldplay"]
}

seed_data = [(artist, genre) for genre, artists in seed_genres_f0.items() for artist in artists]
seed_df = spark.createDataFrame(seed_data, ["seed_artist", "genre"])

user_exposure = clean_df.join(
    seed_df,
    clean_df.artist_name == seed_df.seed_artist,
    "inner"
).select("user_id", "genre").distinct()

user_genre_counts = user_exposure.groupBy("user_id").count()
pure_users = user_genre_counts.filter(F.col("count") == 1).select("user_id")
user_exposure_pure = user_exposure.join(pure_users, "user_id", "inner")

expanded_candidates = user_exposure_pure.join(clean_df, "user_id", "inner")
candidate_counts = expanded_candidates.groupBy("genre", "artist_name") \
    .count().withColumnRenamed("count", "genre_count")

global_counts = clean_df.groupBy("artist_name") \
    .count().withColumnRenamed("count", "global_count")

MIN_GENRE_COUNT = 20
MIN_GLOBAL_COUNT = 50

scored_candidates = candidate_counts.join(global_counts, "artist_name") \
    .withColumn("relevance_score", F.col("genre_count") / F.col("global_count")) \
    .filter(F.col("genre_count") >= MIN_GENRE_COUNT) \
    .filter(F.col("global_count") >= MIN_GLOBAL_COUNT)

f0_artist_list = [artist for artists in seed_genres_f0.values() for artist in artists]
new_candidates = scored_candidates.filter(~F.col("artist_name").isin(f0_artist_list))

window_spec = Window.partitionBy("genre").orderBy(
    F.col("relevance_score").desc(),
    F.col("genre_count").desc()
)
top_f1_artists = new_candidates.withColumn("rank", F.row_number().over(window_spec)) \
                               .filter(F.col("rank") <= 50) \
                               .select("genre", "artist_name")

f1_seed_df = top_f1_artists.groupBy("genre").agg(F.collect_list("artist_name").alias("f1_artists"))
f1_seed_dict = {row['genre']: row['f1_artists'] for row in f1_seed_df.collect()}

final_seeds = {}
print("\n🌟 KẾT QUẢ KHUẾCH ĐẠI HẠT GIỐNG:")
for genre in seed_genres_f0.keys():
    combined = seed_genres_f0[genre] + f1_seed_dict.get(genre, [])
    final_seeds[genre] = combined
    print(f"   => Nhóm {genre}: {len(combined)} Hạt giống (VD mới: {combined[len(seed_genres_f0[genre]):len(seed_genres_f0[genre])+4]}...)")

print("\n🎉 HOÀN TẤT TRẠM 1!")

# Lưu checkpoint dạng Parquet — không dùng toPandas() tránh OOM
filtered_users_df.write.mode("overwrite").parquet("/kaggle/working/ckpt_filtered_users")
clean_df.write.mode("overwrite").parquet("/kaggle/working/ckpt_clean_df")
with open("/kaggle/working/ckpt_final_seeds.json", "w") as f:
    json.dump(final_seeds, f)
print("💾 Đã lưu checkpoint Trạm 1!")

In [ ]:
from pyspark.ml.feature import CountVectorizer, MinHashLSH
import pyspark.sql.functions as F
from pyspark.sql.types import IntegerType

print("\n🚀 BẮT ĐẦU TRẠM 2: CHUYỂN ĐỔI SANG VECTOR VÀ BĂM DỮ LIỆU LSH...")

cv = CountVectorizer(inputCol="artist_set", outputCol="features", minDF=3)
print("⏳ Đang Vector hóa dữ liệu...")
cv_model = cv.fit(filtered_users_df)
vectorized_df = cv_model.transform(filtered_users_df)

@F.udf(returnType=IntegerType())
def count_nonzeros(v):
    return v.numNonzeros()

valid_vectorized_df = vectorized_df.filter(count_nonzeros(F.col("features")) > 0).cache()
print("✅ Đã dọn dẹp các Vector rỗng!")

mh = MinHashLSH(inputCol="features", outputCol="hashes", numHashTables=5)
print("⏳ Đang tạo Chữ ký số (Signature)...")
mh_model = mh.fit(valid_vectorized_df)
users_with_hashes = mh_model.transform(valid_vectorized_df)

print("⏳ Đang kích hoạt LSH để nối Cạnh Đồ thị...")
threshold = 0.8
edges_df = mh_model.approxSimilarityJoin(
    users_with_hashes, users_with_hashes, threshold, distCol="JaccardDistance"
).select(
    F.col("datasetA.user_id").alias("src"),
    F.col("datasetB.user_id").alias("dst"),
    (1 - F.col("JaccardDistance")).alias("weight")
)

# Persist trước khi lưu
final_edges_df = edges_df.filter(F.col("src") < F.col("dst")) \
                         .filter(F.col("weight") >= 0.2) \
                         .persist()

# Dùng limit().toPandas() thay show() tránh OOM
print("✅ Đã dựng xong Mạng lưới! 5 Cạnh đầu tiên:")
print(final_edges_df.limit(5).toPandas().to_string(index=False))

# Lưu checkpoint dạng Parquet
final_edges_df.write.mode("overwrite").parquet("/kaggle/working/ckpt_edges")
print("💾 Đã lưu checkpoint Trạm 2!")
print("\n🎉 HOÀN TẤT TRẠM 2!")

In [ ]:
import pyspark.sql.functions as F
from pyspark.sql.types import FloatType, ArrayType
from pyspark.sql.window import Window
from functools import reduce

print("\n🚀 BẮT ĐẦU TRẠM 3: LABEL PROPAGATION QUA ĐỒ THỊ...")

spark.sparkContext.setCheckpointDir("/kaggle/working/checkpoints")

GENRES = ["POP", "HIPHOP", "EDM", "RNB", "ROCK"]

# BƯỚC 1: GẮN NHÃN HẠT GIỐNG
print("⏳ Đang gắn nhãn Hạt giống cho các Node...")

seed_labels = []
for genre, artists in final_seeds.items():
    matched = clean_df.filter(F.col("artist_name").isin(artists)) \
                      .select("user_id").distinct() \
                      .withColumn("seed_genre", F.lit(genre))
    seed_labels.append(matched)

# Persist seed_labels_df — dùng lại ở Trạm 4
seed_labels_df = reduce(lambda a, b: a.union(b), seed_labels).persist()

seed_genre_counts = seed_labels_df.groupBy("user_id").count()
pure_seed_users = seed_genre_counts.filter(F.col("count") == 1).select("user_id")
seed_labels_pure_df = seed_labels_df.join(pure_seed_users, "user_id", "inner")

print(f"✅ Số User được gắn nhãn Hạt giống: {seed_labels_pure_df.count()}")

# BƯỚC 2: KHỞI TẠO VECTOR XÁC SUẤT
print("⏳ Đang khởi tạo Vector xác suất ban đầu...")

all_node_ids = final_edges_df.select(F.col("src").alias("user_id")) \
    .union(final_edges_df.select(F.col("dst").alias("user_id"))) \
    .distinct()

nodes_with_seed = all_node_ids.join(seed_labels_pure_df, "user_id", "left")

def init_prob_vector(genre):
    if genre is None:
        return [0.2, 0.2, 0.2, 0.2, 0.2]
    vec = [0.0] * 5
    vec[GENRES.index(genre)] = 1.0
    return vec

init_udf = F.udf(init_prob_vector, ArrayType(FloatType()))

current_nodes = nodes_with_seed \
    .withColumn("prob_vector", init_udf(F.col("seed_genre"))) \
    .withColumn("is_seed", F.col("seed_genre").isNotNull()) \
    .select("user_id", "prob_vector", "is_seed", "seed_genre") \
    .checkpoint()
print("✅ Đã khởi tạo Vector xác suất!")

# BƯỚC 3: LAN TRUYỀN NHÃN
print("⏳ Đang lan truyền Nhãn qua Đồ thị...")

ALPHA = 0.7
NUM_ITERATIONS = 5

edges_bidirectional = final_edges_df.select(
    F.col("src").alias("user_id"),
    F.col("dst").alias("neighbor_id"),
    F.col("weight")
).union(
    final_edges_df.select(
        F.col("dst").alias("user_id"),
        F.col("src").alias("neighbor_id"),
        F.col("weight")
    )
).persist()

for iteration in range(NUM_ITERATIONS):
    neighbor_vectors = edges_bidirectional.join(
        current_nodes.select("user_id", "prob_vector"),
        edges_bidirectional.neighbor_id == current_nodes.user_id,
        "inner"
    ).select(
        edges_bidirectional.user_id,
        F.col("weight"),
        F.col("prob_vector").alias("neighbor_prob")
    )

    agg_df = neighbor_vectors.groupBy("user_id").agg(
        F.sum("weight").alias("total_weight"),
        *[F.sum(F.col("neighbor_prob")[i] * F.col("weight")).alias(f"sum_g{i}") for i in range(5)]
    )

    neighbor_mean_df = agg_df.withColumn(
        "neighbor_mean",
        F.array(*[F.col(f"sum_g{i}") / F.col("total_weight") for i in range(5)])
    ).select("user_id", "neighbor_mean")

    updated = current_nodes.join(neighbor_mean_df, "user_id", "left")

    def blend_vectors(is_seed, prob_vec, neighbor_mean):
        if neighbor_mean is None:
            return prob_vec
        if is_seed:
            return [ALPHA * n + (1 - ALPHA) * p for p, n in zip(prob_vec, neighbor_mean)]
        else:
            return list(neighbor_mean)

    blend_udf = F.udf(blend_vectors, ArrayType(FloatType()))

    current_nodes = updated.withColumn(
        "prob_vector",
        blend_udf(F.col("is_seed"), F.col("prob_vector"), F.col("neighbor_mean"))
    ).select("user_id", "prob_vector", "is_seed", "seed_genre") \
     .checkpoint()

    print(f"   ✅ Vòng {iteration + 1}/{NUM_ITERATIONS} hoàn tất")

print("✅ Lan truyền xong!")

# BƯỚC 4: XUẤT KẾT QUẢ
print("⏳ Đang đóng gói Kết quả...")

result_df = current_nodes.select(
    "user_id",
    *[F.round(F.col("prob_vector")[i] * 100, 1).alias(f"{GENRES[i]}_%") for i in range(5)]
)

result_df = result_df.withColumn(
    "max_val", F.greatest(*[F.col(f"{g}_%") for g in GENRES])
).withColumn(
    "Dominant_Genre",
    F.when(F.col("POP_%") == F.col("max_val"), "POP")
     .when(F.col("HIPHOP_%") == F.col("max_val"), "HIPHOP")
     .when(F.col("EDM_%") == F.col("max_val"), "EDM")
     .when(F.col("RNB_%") == F.col("max_val"), "RNB")
     .otherwise("ROCK")
).drop("max_val")

user_taste_profile_df = result_df.persist()

print("\n🌟 MẪU KẾT QUẢ USER TASTE PROFILE:")
print(user_taste_profile_df.limit(10).toPandas().to_string(index=False))

print(f"\n📊 Thống kê Dominant Genre:")
print(user_taste_profile_df.groupBy("Dominant_Genre").count()
      .orderBy(F.col("count").desc()).toPandas().to_string(index=False))

# Lưu checkpoint dạng Parquet
user_taste_profile_df.write.mode("overwrite").parquet("/kaggle/working/ckpt_user_taste")
print("💾 Đã lưu checkpoint Trạm 3!")
print("\n🎉 HOÀN TẤT TRẠM 3!")

In [ ]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window
import pandas as pd

print("\n🚀 BẮT ĐẦU TRẠM 4: ĐÓNG GÓI DỮ LIỆU ĐẦU RA...")

GENRES = ["POP", "HIPHOP", "EDM", "RNB", "ROCK"]

# BƯỚC 0: BỔ SUNG NHÃN CHO USER NGOÀI ĐỒ THỊ
print("⏳ Bổ sung nhãn cho User ngoài đồ thị...")

users_in_graph = user_taste_profile_df.select("user_id")
all_users = filtered_users_df.select("user_id")
users_outside_graph = all_users.join(users_in_graph, "user_id", "left_anti")

outside_with_seed = users_outside_graph.join(seed_labels_df, "user_id", "inner")

outside_genre_counts = outside_with_seed.groupBy("user_id", "seed_genre").count()
outside_total = outside_genre_counts.groupBy("user_id") \
    .agg(F.sum("count").alias("total_seeds"))
outside_ratios = outside_genre_counts.join(outside_total, "user_id") \
    .withColumn("ratio", F.round((F.col("count") / F.col("total_seeds")) * 100, 1))

outside_profiles = outside_ratios.groupBy("user_id").pivot(
    "seed_genre", GENRES
).agg(F.first("ratio")).fillna(0.0)

for g in GENRES:
    if g in outside_profiles.columns:
        outside_profiles = outside_profiles.withColumnRenamed(g, f"{g}_%")
    else:
        outside_profiles = outside_profiles.withColumn(f"{g}_%", F.lit(0.0))

outside_profiles = outside_profiles.withColumn(
    "max_val", F.greatest(*[F.col(f"{g}_%") for g in GENRES])
).withColumn(
    "Dominant_Persona",
    F.when(F.col("POP_%") == F.col("max_val"), "POP")
     .when(F.col("HIPHOP_%") == F.col("max_val"), "HIPHOP")
     .when(F.col("EDM_%") == F.col("max_val"), "EDM")
     .when(F.col("RNB_%") == F.col("max_val"), "RNB")
     .otherwise("ROCK")
).withColumn(
    "lsh_bucket_id", F.md5(F.col("user_id").cast("string"))
).drop("max_val")

print(f"✅ User ngoài đồ thị có seed: {outside_profiles.count():,}")

# BƯỚC 1: GỘP PROFILE LPA + OUTSIDE
user_taste_final = user_taste_profile_df \
    .withColumnRenamed("Dominant_Genre", "Dominant_Persona") \
    .join(
        users_with_hashes.select(
            "user_id",
            F.md5(F.col("user_id").cast("string")).alias("lsh_bucket_id")
        ), "user_id", "left"
    )

user_taste_combined = user_taste_final.union(
    outside_profiles.select(user_taste_final.columns)
).persist()

# ĐẦU RA 1: ARTIST GENRE PROFILE
print("\n⏳ Đang tạo Bảng 1: Artist Genre Profile...")

user_taste_labeled = user_taste_combined.filter(F.col("POP_%") != 20.0)

artist_fan_profile = clean_df.join(
    user_taste_labeled.select("user_id", *[f"{g}_%" for g in GENRES]),
    "user_id", "inner"
)

artist_genre_profile = artist_fan_profile.groupBy("artist_name").agg(
    F.count("user_id").alias("fan_count"),
    *[F.round(F.avg(f"{g}_%"), 1).alias(f"{g}_%") for g in GENRES]
).filter(F.col("fan_count") >= 10)

artist_genre_profile = artist_genre_profile.withColumn(
    "max_val", F.greatest(*[F.col(f"{g}_%") for g in GENRES])
).withColumn(
    "Dominant_Genre",
    F.when(F.col("POP_%") == F.col("max_val"), "POP")
     .when(F.col("HIPHOP_%") == F.col("max_val"), "HIPHOP")
     .when(F.col("EDM_%") == F.col("max_val"), "EDM")
     .when(F.col("RNB_%") == F.col("max_val"), "RNB")
     .otherwise("ROCK")
).drop("max_val")

artist_genre_profile_pd = artist_genre_profile \
    .orderBy(F.col("fan_count").desc()).toPandas()
artist_genre_profile_pd.to_csv(
    "/kaggle/working/artist_genre_profile.csv",
    index=False, encoding="utf-8-sig"
)
print(f"✅ Artist Genre Profile: {len(artist_genre_profile_pd):,} nghệ sĩ")
print(artist_genre_profile_pd.head(10).to_string(index=False))

# ĐẦU RA 2: USER TASTE PROFILE
print("\n⏳ Đang lưu Bảng 2: User Taste Profile...")
user_taste_combined_pd = user_taste_combined.toPandas()
user_taste_combined_pd.to_csv(
    "/kaggle/working/user_taste_profile.csv",
    index=False, encoding="utf-8-sig"
)
print(f"✅ User Taste Profile: {len(user_taste_combined_pd):,} users")

print("\n" + "="*60)
print("📊 TỔNG KẾT TRẠM 4:")
print(f"   - Số Nghệ sĩ đã phân tích : {len(artist_genre_profile_pd):,}")
print(f"   - Số User đã phân tích     : {len(user_taste_combined_pd):,}")
print(f"   - File xuất ra             : /kaggle/working/")
print("\n   📂 Phân bố Dominant Persona:")
print(user_taste_combined_pd["Dominant_Persona"].value_counts().to_string())
print("="*60)
print("\n🎉 HOÀN TẤT TRẠM 4! Dữ liệu đã sẵn sàng cho Streamlit Dashboard.")

In [ ]:
# =====================================================================
# TRẠM 5: ĐÁNH GIÁ ĐỊNH LƯỢNG CHẤT LƯỢNG PHÂN CỤM
# =====================================================================
print("\n🚀 BẮT ĐẦU TRẠM 5: ĐÁNH GIÁ ĐỊNH LƯỢNG...")

import pandas as pd
import numpy as np
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.preprocessing import StandardScaler

GENRES = ["POP", "HIPHOP", "EDM", "RNB", "ROCK"]
genre_cols = [f"{g}_%" for g in GENRES]

# =====================================================================
# BƯỚC 1: CHUẨN BỊ DỮ LIỆU
# Chỉ lấy user có profile thực (loại 20/20/20/20/20)
# =====================================================================
print("⏳ Chuẩn bị dữ liệu đánh giá...")

df = user_taste_combined_pd.copy()
df_labeled = df[df["POP_%"] != 20.0].copy()

# Vector đặc trưng
X = df_labeled[genre_cols].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Nhãn số
label_map = {g: i for i, g in enumerate(GENRES)}
y = df_labeled["Dominant_Persona"].map(label_map).values

print(f"✅ Số user đưa vào đánh giá: {len(df_labeled):,}")

# =====================================================================
# BƯỚC 2: SILHOUETTE SCORE
# Đo độ gắn kết nội cụm và độ tách biệt giữa các cụm
# Thang: -1 → 1, càng cao càng tốt, > 0.5 là tốt
# =====================================================================
print("\n⏳ Đang tính Silhouette Score...")

# Sample 5000 user để tránh tính quá lâu
sample_size = min(5000, len(df_labeled))
idx = np.random.choice(len(X_scaled), sample_size, replace=False)
X_sample = X_scaled[idx]
y_sample = y[idx]

sil_score = silhouette_score(X_sample, y_sample, metric="euclidean")
print(f"✅ Silhouette Score: {sil_score:.4f}")

# =====================================================================
# BƯỚC 3: DAVIES-BOULDIN INDEX
# Đo độ chồng chéo giữa các cụm
# Thang: càng nhỏ càng tốt, < 1.0 là tốt
# =====================================================================
print("⏳ Đang tính Davies-Bouldin Index...")
db_score = davies_bouldin_score(X_scaled, y)
print(f"✅ Davies-Bouldin Index: {db_score:.4f}")

# =====================================================================
# BƯỚC 4: CALINSKI-HARABASZ SCORE
# Đo tỷ lệ phương sai giữa cụm / trong cụm
# Thang: càng cao càng tốt
# =====================================================================
print("⏳ Đang tính Calinski-Harabasz Score...")
ch_score = calinski_harabasz_score(X_scaled, y)
print(f"✅ Calinski-Harabasz Score: {ch_score:.2f}")

# =====================================================================
# BƯỚC 5: PURITY SCORE
# % user được gán đúng genre so với genre chiếm đa số trong cụm
# =====================================================================
print("⏳ Đang tính Purity Score...")

def purity_score(labels, predictions):
    from collections import Counter
    total = len(labels)
    correct = 0
    for genre in set(predictions):
        mask = predictions == genre
        if mask.sum() > 0:
            most_common = Counter(labels[mask]).most_common(1)[0][1]
            correct += most_common
    return correct / total

purity = purity_score(y, y)  # So sánh với chính nhãn dominant
print(f"✅ Purity Score: {purity:.4f}")

# =====================================================================
# BƯỚC 6: PHÂN TÍCH ĐỘ THUẦN KHIẾT TỪNG CỤM
# =====================================================================
print("\n⏳ Đang phân tích độ thuần khiết từng cụm...")

cluster_stats = []
for genre in GENRES:
    subset = df_labeled[df_labeled["Dominant_Persona"] == genre]
    if len(subset) == 0:
        continue
    dominant_col = f"{genre}_%"
    avg_dominant = subset[dominant_col].mean()
    avg_others = subset[[c for c in genre_cols if c != dominant_col]].mean().mean()
    separation = avg_dominant - avg_others
    cluster_stats.append({
        "Genre": genre,
        "Số User": len(subset),
        "% Dominant TB": round(avg_dominant, 1),
        "% Khác TB": round(avg_others, 1),
        "Độ tách biệt": round(separation, 1)
    })

cluster_df = pd.DataFrame(cluster_stats)
print(cluster_df.to_string(index=False))

# =====================================================================
# TỔNG KẾT ĐÁNH GIÁ
# =====================================================================
print("\n" + "="*60)
print("📊 TỔNG KẾT ĐÁNH GIÁ ĐỊNH LƯỢNG:")
print(f"   Silhouette Score      : {sil_score:.4f}  (tốt nếu > 0.5)")
print(f"   Davies-Bouldin Index  : {db_score:.4f}  (tốt nếu < 1.0)")
print(f"   Calinski-Harabasz     : {ch_score:.2f}  (càng cao càng tốt)")

# Đánh giá tổng thể
if sil_score > 0.5 and db_score < 1.0:
    verdict = "🟢 TỐT — Các cụm tách biệt rõ ràng"
elif sil_score > 0.3:
    verdict = "🟡 TRUNG BÌNH — Các cụm có xu hướng tách biệt"
else:
    verdict = "🔴 YẾU — Các cụm còn chồng chéo nhiều"

print(f"\n   Đánh giá tổng thể: {verdict}")
print("="*60)
print("\n🎉 HOÀN TẤT ĐÁNH GIÁ ĐỊNH LƯỢNG!")

# Lưu kết quả đánh giá
eval_results = {
    "silhouette_score": round(sil_score, 4),
    "davies_bouldin_index": round(db_score, 4),
    "calinski_harabasz_score": round(ch_score, 2),
    "total_users_evaluated": len(df_labeled),
    "cluster_stats": cluster_df.to_dict(orient="records")
}
import json
with open("/kaggle/working/evaluation_results.json", "w") as f:
    json.dump(eval_results, f, indent=2, ensure_ascii=False)
print("💾 Đã lưu kết quả đánh giá!")